In [ ]:
!pip install torch torchvision -q

In [ ]:
!python dataset.py


Dataset
  Entities (N):                    1000
  Relations (R):                   11  →  N×R = 11000
  Extraction relations:            ['class', 'color', 'material', 'origin', 'shape', 'size']
  Composition relations:           ['same_color_as', 'same_shape_as', 'same_size_as', 'same_material_as', 'same_origin_as']
  Training queries:                6000
  Test queries (N×R):              6000

Sample entity:
  {
    "name": "Bliblax-210",
    "class": "warrior",
    "color": "green",
    "material": "wood",
    "origin": "region_B",
    "shape": "star",
    "size": "huge"
}

Training sequences for Bliblax-210:
  [extraction] <S> Bliblax-210 </S> <R> class </R> <O> warrior </O>
  [extraction] <S> Bliblax-210 </S> <R> color </R> <O> green </O>
  [extraction] <S> Bliblax-210 </S> <R> material </R> <O> wood </O>
  [extraction] <S> Bliblax-210 </S> <R> origin </R> <O> region_B </O>
  [extraction] <S> Bliblax-210 </S> <R> shape </R> <O> star </O>
  [extraction] <S> Bliblax-210 </S> <R> s

In [ ]:
!python tokenizer.py

  Loaded 1000 entity names from dataset_extraction.json
SROTokenizer built:
  Special tokens:   8
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1051
  Tokenizer saved → tokenizer.json

── Encoding examples ──

  Input:   <S> Bliblax-210 </S> <R> color </R> <O> blue </O>
  Tokens:  ['<S>', 'Bliblax-210', '</S>', '<R>', 'color', '</R>', '<O>', 'blue', '</O>']
  IDs:     [2, 51, 3, 4, 45, 5, 6, 9, 7]
  Decoded: <S> Bliblax-210 </S> <R> color </R> <O> blue </O>

  Input:   <S> Bliforn-232 </S> <R> same color as </R> <O> Blimpnik-34 </O>
  Tokens:  ['<S>', 'Bliforn-232', '</S>', '<R>', 'same', 'color', 'as', '</R>', '<O>', 'Blimpnik-34', '</O>']
  IDs:     [2, 56, 3, 4, 48, 45, 43, 5, 6, 1, 7]
  Decoded: <S> Bliforn-232 </S> <R> same color as </R> <O> <UNK> </O>

  Input:   <S> Bligast-300 </S> <R> material </R> <O> metal </O>
  Tokens:  ['<S>', 'Bligast-300', '</S>', '<R>', 'material', '</R>', '<O>', 'metal', '</O>']
  IDs:  

In [ ]:
!python train_experiment.py --model both --out ./outputs

Device: cuda
  Tokenizer loaded from tokenizer.json  (vocab size: 1051)

Building dataloaders...
  Train sequences: 6000  (188 batches)

Training: SUMMED (baseline)
  [summed] 3,702,784 parameters  | 4 blocks | 4 heads
  [summed] early stopping: patience=20, eval_every=500
  [summed] step    200 | loss 1.334275 | lr 9.95e-04
  [summed] step    400 | loss 1.324787 | lr 1.00e-03
  [summed] step    500 | loss 1.347129  ✓ best
  [summed] step    600 | loss 1.339946 | lr 1.00e-03
  [summed] step    800 | loss 1.280505 | lr 9.99e-04
  [summed] step   1000 | loss 1.276273 | lr 9.98e-04
  [summed] step   1000 | loss 1.276273  ✓ best
  [summed] step   1200 | loss 1.240006 | lr 9.97e-04
  [summed] step   1400 | loss 1.245192 | lr 9.96e-04
  [summed] step   1500 | loss 1.204495  ✓ best
  [summed] step   1600 | loss 1.177615 | lr 9.95e-04
  [summed] step   1800 | loss 1.171826 | lr 9.93e-04
  [summed] step   2000 | loss 1.169687 | lr 9.91e-04
  [summed] step   2000 | loss 1.169687  ✓ best
  [summe

In [ ]:
!python eval_extraction.py

  Loaded 1000 entity names from dataset_extraction.json
SROTokenizer built:
  Special tokens:   8
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1051
  [summed] 3,702,784 parameters  | 4 blocks | 4 heads
  [disentangled] 3,631,680 parameters  | 4 blocks | 4 heads
Evaluating summed model...

────────────────────────────────────────────────────────────
Subject : Bliblax-210
Relation: class
Prompt  : <S> Bliblax-210 </S> <R> class </R> <O>
Gold    : warrior </O>
Pred    : warrior </O>
Correct : ✓

────────────────────────────────────────────────────────────
Subject : Bliblax-210
Relation: color
Prompt  : <S> Bliblax-210 </S> <R> color </R> <O>
Gold    : green </O>
Pred    : green </O>
Correct : ✓

────────────────────────────────────────────────────────────
Subject : Bliblax-210
Relation: material
Prompt  : <S> Bliblax-210 </S> <R> material </R> <O>
Gold    : wood </O>
Pred    : wood </O>
Correct : ✓

────────────────────────

In [ ]:
!python evaluate.py \
--summed outputs/summed/model_best.pt \
--disentangled outputs/disentangled/model_best.pt \
--dataset dataset.json \
--out eval_results.json

Loading dataset...
  Loaded 1000 entity names from dataset.json
SROTokenizer built:
  Special tokens:   9
  Attribute values: 35
  Relation words:   8
  Entity names:     1000  (from dataset.json)
  Total vocab:      1052
Building valid answer sets for composition queries...

  Valid answer counts for 'Bliblax-210':
    same_color_as          → 128 valid answers  (attr=green)
    same_shape_as          → 140 valid answers  (attr=star)
    same_size_as           → 216 valid answers  (attr=huge)
    same_material_as       → 154 valid answers  (attr=wood)
    same_origin_as         → 183 valid answers  (attr=region_B)

Loading models...
  [summed] 3,730,944 parameters  | 4 blocks | 4 heads
  [disentangled] 3,639,040 parameters  | 4 blocks | 4 heads

──────────────────────────────────────────────────
Evaluating: summed
──────────────────────────────────────────────────
① N×R accuracy...
② Compositional generalization...

════════════════════════════════════════════════════════════
 summed
